In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [2]:
from pathlib import Path

PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

for folder in [
    RAW_DIR,
    PROCESSED_DIR,
    TABLE_DIR,
    FIGURE_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "Project root:",
    PROJECT_ROOT.resolve()
)

Project root: C:\quant_edge_round2


In [3]:
print("Current working directory:")
print(Path.cwd())

Current working directory:
C:\quant_edge_round2\notebooks


In [4]:
RAW_PRICE_FILE = RAW_DIR / "prices_raw.csv"
COMMON_PRICE_FILE = PROCESSED_DIR / "prices_common.csv"

DATA_SUMMARY_FILE = TABLE_DIR / "data_summary.csv"
QUALITY_FLAGS_FILE = TABLE_DIR / "data_quality_flags.csv"

NORMALIZED_PRICE_FIGURE = FIGURE_DIR / "normalized_prices.png"

In [5]:
TICKERS = [
    "SPY",
    "EFA",
    "EEM",
    "AGG",
    "GLD",
    "XLE",
    "XLK",
]

ASSET_NAMES = {
    "SPY": "US Large-Cap Equities",
    "EFA": "International Developed Equities",
    "EEM": "Emerging Market Equities",
    "AGG": "US Investment-Grade Bonds",
    "GLD": "Gold / Real Asset",
    "XLE": "Energy Sector",
    "XLK": "Technology Sector",
}

In [6]:
START_DATE = "2000-01-01"
END_DATE = "2026-10-20"

In [7]:
if RAW_PRICE_FILE.exists():
    print("Loading saved raw prices...")
    prices_raw = pd.read_csv(
        RAW_PRICE_FILE,
        index_col=0,
        parse_dates=True
    )

else:
    print("Downloading prices from Yahoo Finance...")

    downloaded = yf.download(
        TICKERS,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=True,
        progress=False,
        group_by="column",
        threads=False
    )

    if downloaded.empty:
        raise ValueError("Yahoo Finance returned an empty dataset.")

    if isinstance(downloaded.columns, pd.MultiIndex):
        prices_raw = downloaded["Close"].copy()
    else:
        prices_raw = downloaded[["Close"]].copy()
        prices_raw.columns = TICKERS

    prices_raw = prices_raw.sort_index()

    prices_raw.to_csv(RAW_PRICE_FILE)

    print(f"Raw prices saved to: {RAW_PRICE_FILE}")

Loading saved raw prices...


In [8]:
prices_raw.index = pd.to_datetime(
    prices_raw.index
)

prices_raw = (
    prices_raw
    .sort_index()
)

if END_DATE is not None:

    prices_raw = prices_raw.loc[
        prices_raw.index
        < pd.Timestamp(
            END_DATE
        )
    ]

print(
    "Analysis cutoff applied."
)

print(
    "Last raw observation:",
    prices_raw.index.max()
)

Analysis cutoff applied.
Last raw observation: 2026-10-19 00:00:00


In [9]:
prices_raw = prices_raw.reindex(columns=TICKERS)
prices_raw.index = pd.to_datetime(prices_raw.index)
prices_raw = prices_raw.sort_index()

print(prices_raw.head())
print()
print(prices_raw.tail())
print()
print("Raw shape:", prices_raw.shape)

                  SPY  EFA  EEM  AGG  GLD       XLE        XLK
2000-01-03  90.907005  NaN  NaN  NaN  NaN  6.569213  20.577803
2000-01-04  87.351959  NaN  NaN  NaN  NaN  6.445556  19.533834
2000-01-05  87.508247  NaN  NaN  NaN  NaN  6.615585  19.243849
2000-01-06  86.101868  NaN  NaN  NaN  NaN  6.870623  18.605862
2000-01-07  91.102341  NaN  NaN  NaN  NaN  6.944045  18.930655

                   SPY         EFA        EEM        AGG         GLD  \
2026-10-13  778.010010  103.470001  65.739998  94.709999  381.670013   
2026-10-14  769.599976  102.750000  64.389999  94.820000  375.989990   
2026-10-15  771.789978  103.669998  63.959999  95.410004  382.709992   
2026-10-16  763.250000  103.550003  62.849998  95.379997  382.010010   
2026-10-19  767.008972  103.014999  63.360001  95.095001  379.862000   

                  XLE         XLK  
2026-10-13  67.199997  196.279999  
2026-10-14  67.629997  190.429993  
2026-10-15  66.559998  189.190002  
2026-10-16  68.239998  183.470001  
2026-10-

In [10]:
summary_rows = []

for ticker in TICKERS:
    series = prices_raw[ticker]

    valid = series.dropna()

    summary_rows.append({
        "Ticker": ticker,
        "Asset": ASSET_NAMES[ticker],
        "Start Date": valid.index.min() if not valid.empty else pd.NaT,
        "End Date": valid.index.max() if not valid.empty else pd.NaT,
        "Valid Observations": valid.shape[0],
        "Missing Values": series.isna().sum(),
        "Duplicate Dates": prices_raw.index.duplicated().sum(),
    })

data_summary = pd.DataFrame(summary_rows)

data_summary

,Ticker,Asset,Start Date,End Date,Valid Observations,Missing Values,Duplicate Dates
0,SPY,US Large-Cap Equities,2000-01-03,2026-10-19,6739,0,0
1,EFA,International Developed Equities,2001-08-27,2026-10-19,6323,416,0
2,EEM,Emerging Market Equities,2003-04-14,2026-10-19,5917,822,0
3,AGG,US Investment-Grade Bonds,2003-09-29,2026-10-19,5801,938,0
4,GLD,Gold / Real Asset,2004-11-18,2026-10-19,5513,1226,0
5,XLE,Energy Sector,2000-01-03,2026-10-19,6739,0,0
6,XLK,Technology Sector,2000-01-03,2026-10-19,6739,0,0


In [11]:
data_summary.to_csv(DATA_SUMMARY_FILE, index=False)

In [12]:
duplicate_dates = prices_raw.index[prices_raw.index.duplicated()]

print("Number of duplicate dates:", len(duplicate_dates))

if len(duplicate_dates) > 0:
    print("Duplicate dates:")
    print(duplicate_dates)

Number of duplicate dates: 0


In [13]:
non_positive_flags = []

for ticker in TICKERS:
    mask = prices_raw[ticker] <= 0

    for date, value in prices_raw.loc[mask, ticker].items():
        non_positive_flags.append({
            "Date": date,
            "Ticker": ticker,
            "Flag": "Non-positive price",
            "Value": value,
            "Decision": "Review"
        })

print("Non-positive price flags:", len(non_positive_flags))

Non-positive price flags: 0


In [14]:
missing_summary = pd.DataFrame({
    "Missing Values": prices_raw.isna().sum(),
    "Missing Percentage": prices_raw.isna().mean() * 100
})

missing_summary

,Missing Values,Missing Percentage
SPY,0,0.000000
EFA,416,6.173023
EEM,822,12.197655
AGG,938,13.918979
GLD,1226,18.192610
XLE,0,0.000000
XLK,0,0.000000


In [15]:
internal_missing_rows = []

for ticker in TICKERS:
    series = prices_raw[ticker]

    first_valid = series.first_valid_index()
    last_valid = series.last_valid_index()

    active_period = series.loc[first_valid:last_valid]

    internal_missing = active_period.isna().sum()

    internal_missing_rows.append({
        "Ticker": ticker,
        "First Valid Date": first_valid,
        "Last Valid Date": last_valid,
        "Internal Missing Values": internal_missing
    })

internal_missing_summary = pd.DataFrame(internal_missing_rows)

internal_missing_summary

,Ticker,First Valid Date,Last Valid Date,Internal Missing Values
0,SPY,2000-01-03,2026-10-19,0
1,EFA,2001-08-27,2026-10-19,0
2,EEM,2003-04-14,2026-10-19,0
3,AGG,2003-09-29,2026-10-19,0
4,GLD,2004-11-18,2026-10-19,0
5,XLE,2000-01-03,2026-10-19,0
6,XLK,2000-01-03,2026-10-19,0


In [16]:
internal_missing_summary.to_csv(
    TABLE_DIR / "internal_missing_summary.csv",
    index=False
)

In [17]:
simple_returns_check = prices_raw.pct_change(fill_method=None)

LARGE_MOVE_THRESHOLD = 0.20

In [18]:
large_move_flags = []

for ticker in TICKERS:
    mask = simple_returns_check[ticker].abs() > LARGE_MOVE_THRESHOLD

    for date, value in simple_returns_check.loc[mask, ticker].items():
        large_move_flags.append({
            "Date": date,
            "Ticker": ticker,
            "Flag": "Large daily move",
            "Value": value,
            "Decision": "Retain unless confirmed data error"
        })

print("Large-move flags:", len(large_move_flags))

Large-move flags: 3


In [19]:
large_move_df = pd.DataFrame(large_move_flags)

large_move_df

,Date,Ticker,Flag,Value,Decision
0,2008-10-13,EEM,Large daily move,0.227699,Retain unless confirmed data error
1,2008-10-28,EEM,Large daily move,0.209326,Retain unless confirmed data error
2,2020-03-09,XLE,Large daily move,-0.201412,Retain unless confirmed data error


In [20]:
large_move_df["Decision"] = "Retained - genuine market stress observation"

large_move_df

,Date,Ticker,Flag,Value,Decision
0,2008-10-13,EEM,Large daily move,0.227699,Retained - genuine market stress observation
1,2008-10-28,EEM,Large daily move,0.209326,Retained - genuine market stress observation
2,2020-03-09,XLE,Large daily move,-0.201412,Retained - genuine market stress observation


In [21]:
flat_price_flags = []

for ticker in TICKERS:
    unchanged = prices_raw[ticker].eq(prices_raw[ticker].shift(1))

    for date, value in prices_raw.loc[unchanged, ticker].items():
        flat_price_flags.append({
            "Date": date,
            "Ticker": ticker,
            "Flag": "Unchanged price",
            "Value": value,
            "Decision": "Review if repeated"
        })

print("Unchanged-price observations:", len(flat_price_flags))

Unchanged-price observations: 336


In [22]:
stale_run_rows = []

MIN_STALE_RUN = 3

for ticker in TICKERS:
    series = prices_raw[ticker]

    same_as_previous = series.eq(series.shift(1))
    groups = (same_as_previous != same_as_previous.shift()).cumsum()

    for _, group in series.groupby(groups):
        if len(group) >= MIN_STALE_RUN:
            values = group.dropna()

            if len(values) >= MIN_STALE_RUN and values.nunique() == 1:
                stale_run_rows.append({
                    "Ticker": ticker,
                    "Start Date": values.index.min(),
                    "End Date": values.index.max(),
                    "Run Length": len(values),
                    "Price": values.iloc[0]
                })

stale_run_df = pd.DataFrame(stale_run_rows)

print("Potential stale-price runs:", len(stale_run_df))

stale_run_df

Potential stale-price runs: 0


""


In [23]:
prices_common = prices_raw.dropna(subset=TICKERS).copy()

rows_removed = len(prices_raw) - len(prices_common)
removed_pct = rows_removed / len(prices_raw) * 100

print("Raw dataset shape:", prices_raw.shape)
print("Common-date dataset shape:", prices_common.shape)
print("Rows excluded from synchronized analysis:", rows_removed)
print(f"Percentage excluded: {removed_pct:.2f}%")
print("Common start date:", prices_common.index.min())
print("Common end date:", prices_common.index.max())
print("Common observations:", len(prices_common))
print("Number of assets:", prices_common.shape[1])

Raw dataset shape: (6739, 7)
Common-date dataset shape: (5513, 7)
Rows excluded from synchronized analysis: 1226
Percentage excluded: 18.19%
Common start date: 2004-11-18 00:00:00
Common end date: 2026-10-19 00:00:00
Common observations: 5513
Number of assets: 7


In [24]:
quality_flags = large_move_df.copy()

quality_flags.to_csv(
    QUALITY_FLAGS_FILE,
    index=False
)

quality_flags

,Date,Ticker,Flag,Value,Decision
0,2008-10-13,EEM,Large daily move,0.227699,Retained - genuine market stress observation
1,2008-10-28,EEM,Large daily move,0.209326,Retained - genuine market stress observation
2,2020-03-09,XLE,Large daily move,-0.201412,Retained - genuine market stress observation


In [25]:
prices_common.to_csv(COMMON_PRICE_FILE)

print(f"Processed common prices saved to: {COMMON_PRICE_FILE}")

Processed common prices saved to: ..\data\processed\prices_common.csv


In [26]:
normalized_prices = (
    prices_common / prices_common.iloc[0]
) * 100

In [27]:
plt.figure(figsize=(12, 6))

for ticker in TICKERS:
    plt.plot(
        normalized_prices.index,
        normalized_prices[ticker],
        label=ticker
    )

plt.title("Growth of 100 Units Invested in Each ETF")
plt.xlabel("Date")
plt.ylabel("Normalized Value (Start = 100)")
plt.legend()
plt.tight_layout()

plt.savefig(
    NORMALIZED_PRICE_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_35168\3342143385.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:
governance_summary = pd.DataFrame({
    "Check": [
        "Internal missing values",
        "Duplicate dates",
        "Non-positive prices",
        "Large move flags",
        "Potential stale-price runs",
        "Common observations"
    ],
    "Result": [
        int(internal_missing_summary["Internal Missing Values"].sum()),
        int(prices_raw.index.duplicated().sum()),
        len(non_positive_flags),
        len(large_move_flags),
        len(stale_run_df),
        len(prices_common)
    ]
})

governance_summary

,Check,Result
0,Internal missing values,0
1,Duplicate dates,0
2,Non-positive prices,0
3,Large move flags,3
4,Potential stale-price runs,0
5,Common observations,5513


In [29]:
governance_summary.to_csv(
    TABLE_DIR / "data_governance_summary.csv",
    index=False
)

In [30]:
assert prices_common.shape[1] == len(TICKERS)
assert prices_common.index.is_unique
assert prices_common.isna().sum().sum() == 0
assert (prices_common > 0).all().all()

assert prices_raw.index.max() < pd.Timestamp(
    END_DATE
)

assert prices_common.index.max() < pd.Timestamp(
    END_DATE
)

print(
    "All final validation checks passed."
)

All final validation checks passed.
